# Streaming, APIs and user interfaces

Assistant version 10: FastAPI streaming backend and Streamlit chat with per-thread memory and approval controls.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra deep --extra deployment --group authoring`, then `uv run --extra deep --extra deployment --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Coroutines, concurrency, cancellation and blocking work; token and step streaming

Predict the output before running. The function definitions below match `examples/01_async_basics.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Await I/O, bound concurrency, cancel work, and inspect model chunks.

Run: uv run --extra apps python modules/18_apps/examples/01_async_basics.py --offline
Expected: Three replies, cancellation acknowledgement and streamed chunks.
Cost: Four short model calls live; zero in offline mode. Requires: apps
"""

import asyncio

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


async def demonstrate(settings) -> None:
    semaphore = asyncio.Semaphore(2)

    async def reply(index: int) -> str:
        async with semaphore:
            model = get_chat_model(settings=settings, responses=[f"Brightwave reply {index}"])
            result = await model.ainvoke("Say a short Brightwave greeting.")
            return str(result.content)

    print_text(str(await asyncio.gather(*(reply(i) for i in range(3)))))
    waiting = asyncio.create_task(asyncio.sleep(60))
    waiting.cancel()
    try:
        await waiting
    except asyncio.CancelledError:
        print_text("Cancelled and awaited; resources can now be released.")
    # KEY IDEA: move blocking work off the event loop, not into an async def unchanged.
    print_text(await asyncio.to_thread(lambda: "Blocking preparation finished."))
    model = get_chat_model(settings=settings, responses=["Brightwave stream complete."])
    async for chunk in model.astream("Say a short Brightwave greeting."):
        print_text(str(chunk.content))


def main() -> None:
    asyncio.run(demonstrate(lesson_settings()))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## FastAPI StreamingResponse; server-sent-event framing; disconnects and errors

Predict the output before running. The function definitions below match `examples/02_sse.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Frame server-sent events and consume an authenticated support stream.

Run: uv run --extra apps python modules/18_apps/examples/02_sse.py --offline
Expected: Escaped framing, progress, assistant text and a done event.
Cost: No API calls; deterministic local checks. Requires: apps
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.release import headers
from common.web import demo_app, sse


def main() -> None:
    lesson_settings()
    print_text(sse("text", {"text": "one\ntwo"}))
    with TemporaryDirectory() as directory, TestClient(demo_app(Path(directory))) as client:
        thread = client.post("/threads", headers=headers()).json()["thread_id"]
        # KEY IDEA: HTTP status validates access before the event stream starts.
        with client.stream(
            "POST",
            f"/threads/{thread}/messages",
            headers=headers(),
            json={"text": "Please check BW1001"},
        ) as response:
            response.raise_for_status()
            for line in response.iter_lines():
                print_text(line)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Streamlit chat; per-conversation thread state; new conversation reset

Predict the output before running. The function definitions below match `examples/03_streamlit_chat.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Render the real Streamlit entry point in its application test harness.

Run: uv run --extra apps python modules/18_apps/examples/03_streamlit_chat.py --offline
Expected: The credential prompt renders; no server or real model is contacted.
Cost: No API calls; deterministic local checks. Requires: apps
"""

from streamlit.testing.v1 import AppTest

from common.config import ROOT
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    # KEY IDEA: AppTest executes the same render function as streamlit run.
    app = AppTest.from_file(ROOT / "modules/18_apps/use_case/app.py")
    AppTest.run(app)
    assert not app.exception
    assert app.title[0].value == "Brightwave support"
    print_text(app.info[0].value)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Approve/reject buttons; interrupted state; server-side authorization and safe resume

Predict the output before running. The function definitions below match `examples/04_approval_ui.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Exercise the HTTP endpoints behind approve and reject buttons.

Run: uv run --extra apps python modules/18_apps/examples/04_approval_ui.py --offline
Expected: Both reviewed outcomes; a customer review and repeated resume are rejected.
Cost: No API calls; deterministic local checks. Requires: apps
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.release import api_contract


def main() -> None:
    lesson_settings()
    for decision in ("approve", "reject"):
        with TemporaryDirectory() as directory:
            # KEY IDEA: a reviewer role and current interrupt ID are checked on the server.
            result = api_contract(Path(directory), decision)
            print_text(str({"decision": decision, **result}))
            assert all(value for key, value in result.items() if key != "result")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Session identity, thread ownership and bounded concurrent requests

Predict the output before running. The function definitions below match `examples/05_session_isolation.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Keep identity in the host and reject overlapping work on the same thread.

Run: uv run --extra apps python modules/18_apps/examples/05_session_isolation.py --offline
Expected: Cross-customer access and overlapping work are denied.
Cost: No API calls; deterministic local checks. Requires: apps
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from common.budget import Admission
from common.config import Settings
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.security import Principal
from common.service import SupportService


def main() -> None:
    lesson_settings()
    with TemporaryDirectory() as directory:
        service = SupportService(Path(directory), Settings(course_offline=True))
        thread = service.new_thread(Principal("C001", "customer"))
        try:
            service.session(thread, Principal("C002", "customer"))
        except PermissionError:
            print_text("Cross-customer access denied.")
    admission = Admission(maximum=1)
    with admission.slot("thread-a", "C001"):
        try:
            # KEY IDEA: a thread identifier is neither a credential nor a lock.
            with admission.slot("thread-a", "C001"):
                raise AssertionError("Overlapping request was admitted")
        except BlockingIOError:
            print_text("Concurrent request rejected; outer slot will be released.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world application entry point

Predict the output before running. The function definitions below match `use_case/api.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Uvicorn factory: uv run --extra apps uvicorn common.web:configured_app --factory.

Run finite check: uv run --extra apps python modules/18_apps/use_case/api.py --offline
Expected: public health and authentication checks. Cost: none. Requires: apps
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.web import demo_app


def main() -> None:
    lesson_settings()
    with TemporaryDirectory() as directory, TestClient(demo_app(Path(directory))) as client:
        assert client.post("/threads").status_code == 401
        print_text(str(client.get("/health").json()))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world application entry point

Predict the output before running. The function definitions below match `use_case/app.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Launch Streamlit with: uv run --extra apps streamlit run modules/18_apps/use_case/app.py.

Run finite check: uv run --extra apps python modules/18_apps/use_case/app.py --offline
Expected: startup guidance. Cost: none for CLI; browser chat may call the configured model.
Requires: apps
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.ui import render


def main() -> None:
    lesson_settings()
    print_text("Start the authenticated API, then run this file with streamlit run.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world application entry point

Predict the output before running. The function definitions below match `use_case/support_web.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Check the guarded local support application end to end.

Run: uv run --extra apps python modules/18_apps/use_case/support_web.py --offline
Expected: authentication, streaming, review and replay checks pass.
Cost: none; this finite demonstration always uses explicit fixtures. Requires: apps
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.release import api_contract


def main() -> None:
    lesson_settings()
    with TemporaryDirectory() as directory:
        result = api_contract(Path(directory))
        assert all(value for key, value in result.items() if key != "result")
        print_text(str(result))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra apps pytest modules/18_apps/tests` for the supplied solution checks.